# Mean Reversion + Dollar Bars

This notebook runs the live mean-reversion engine with:

- underlying signal data from stock bars
- dollar bars as the signal bars
- CatBoost disabled
- configurable stock/option execution

Dollar bars complete when accumulated `close * volume` reaches the configured threshold. With IBKR 5-second bars, a single oversized 5-second bar is kept as one oversized dollar bar because there is no tick-level data to split it precisely.

In [ ]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

from ib_async import IB, Stock

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import (
    active_orders,
    all_orders,
    build_execution_instrument,
    filled_orders,
    print_order_snapshot,
    recent_fills,
    strategy_open_trades,
)
from orders import IBKRLimitOrderRouter
from strategies.live_mean_reversion import LiveMeanReversion


## 1. User Settings

`DOLLAR_THRESHOLD` controls how much traded notional is needed to complete one signal bar.

In [ ]:
SYMBOL = "META"
IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 103

DOLLAR_THRESHOLD = 1_000_000
OPEN_TRADES_PATH = f"{SYMBOL}_dollar_bars_mean_reversion_open_trades.csv"


## 2. Build Config

This example uses `standard_bb` mean reversion on dollar bars. You can switch `entry_strategy.name` to `median_mad` if you want the older median/MAD entry logic on dollar bars.

In [ ]:
raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
raw["symbol"] = SYMBOL
raw["paths"]["open_trades_path"] = OPEN_TRADES_PATH

raw["features"]["bar"] = {
    "type": "dollar",
    "dollar_threshold": DOLLAR_THRESHOLD,
    "history_window": 390,
}

# Feature windows are counted in completed dollar bars.
raw["features"]["window"] = 20
raw["features"]["min_bars"] = 60

# No CatBoost in this example.
raw["model"]["enabled"] = False

# Non-median Bollinger mean reversion on dollar bars.
raw["strategy"]["entry_strategy"] = {
    "name": "standard_bb",
    "z": 2.0,
    "double_down_mult": 3.0,
}

raw["execution"]["instrument"] = {
    "type": "stock",
    "exchange": "SMART",
    "currency": "USD",
    "limit_entry_offset_pct": 0.0005,
    "limit_exit_offset_pct": 0.0005,
}

# Double-down can require underlying and/or execution instrument movement.
raw["double_down"]["price_rules"] = [
    {"basis": "underlying", "mode": "pct", "max_change": -0.01},
]

config = LiveTradingConfig.from_dict(raw)
config.raw["features"]["bar"], config.raw["strategy"]["entry_strategy"]


## 3. Optional: Trade Calls Instead of Stock

Leave this cell unrun if you want stock execution. Run it before building `execution_instrument` if you want call option execution while the signal still uses stock dollar bars.

In [ ]:
# CALL_EXPIRY = "20260619"
# CALL_STRIKE = 500.0
# config.raw["execution"]["instrument"] = {
#     "type": "option",
#     "exchange": "SMART",
#     "currency": "USD",
#     "expiry": CALL_EXPIRY,
#     "strike": CALL_STRIKE,
#     "right": "C",
#     "multiplier": 100.0,
#     "limit_entry_offset_pct": 0.02,
#     "limit_exit_offset_pct": 0.02,
# }


## 4. Connect IBKR and Subscribe to Underlying Stock Bars

The raw IBKR bars are still 5-second bars. The strategy aggregates them into dollar bars.

In [ ]:
ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock = Stock(SYMBOL, "SMART", "USD")
ib.qualifyContracts(stock)

real_time_bars = ib.reqRealTimeBars(
    stock,
    barSize=5,
    whatToShow="TRADES",
    useRTH=False,
)

print("Connected:", ib.isConnected())
print("Signal underlying:", stock)


## 5. Build Strategy

In [ ]:
execution_instrument = build_execution_instrument(
    ib=ib,
    execution_cfg=config.execution,
    symbol=config.symbol,
)

order_router = IBKRLimitOrderRouter(ib=ib, contract=stock)

algo = LiveMeanReversion(
    config=config,
    order_router=order_router,
    execution_instrument=execution_instrument,
)

print("Signal bar builder:", type(algo.signal_bar_builder).__name__)
print("Execution instrument:", execution_instrument.instrument_type)
print("Loaded strategy open trades:", len(algo.open_trades))


## 6. Start / Stop Callback

In [ ]:
real_time_bars.updateEvent += algo.on_bar
print("Attached algo.on_bar")


In [ ]:
real_time_bars.updateEvent -= algo.on_bar
print("Detached algo.on_bar")


## 7. Inspect Dollar Bars and Features

In [ ]:
list(algo.signal_bars)[-5:]


In [ ]:
features = algo.calculate_features()
features


## 8. Order and Trade Views

In [ ]:
print_order_snapshot(ib, algo)


In [ ]:
active_orders(ib)


In [ ]:
filled_orders(ib)


In [ ]:
recent_fills(ib)


In [ ]:
strategy_open_trades(algo)


In [ ]:
all_orders(ib)
